# DP8（本番・Gemma-2-2b・A）— まれな別の語は、出たときどれだけ効くか

**やり方**：**Gemma-2-2b-it** で、算数の問題ごとに貪欲法の推論を1本作り、答えの手前まで **8トークンおき** の位置で、温度1.0の続きを4種類 **32本ずつ** 引きます（土台の語のあと／2番目の候補のあと／**自然に引いた別の語のあと**／確率がほぼ 10⁻⁶ の語のあと）。ゲート：`notes/dp8_rarity_bridge_gate.md`。

**主の問い**
- 主1：自然に引いた別の語は、2番目の候補と同じくらい効くか。
- 主2：別の語が出にくい位置ほど、自然に引いた別の語の効き目は大きいか。
- 主3：まれな位置は、自然な答えのばらつきにほとんど寄与しないか。
- 主4：確率 10⁻⁶ の語は、2番目の候補より多く壊すか。

このノートブックは **行0〜19（40問のうち20問）** を担当します。もう1本 `DP8_GEMMA_B.ipynb` は、**この本が終わってから** 回してね（A → B の順。同時に回しても結果は同じ。結果のフォルダは別々）。マシンの中では、作業プロセスが終わっていない仕事を順に取りに行きます（どのプロセスが担当しても結果のファイルは同じ）。集計は、2本とも終わってから CPU のノートブック `DP8_ANALYZE.ipynb` で行います。

**使い方**
1. ランタイム → ランタイムのタイプを変更 → **L4**（ハイメモリは不要）。
2. Colabの「シークレット」に `GITHUB_TOKEN` と `HF_TOKEN`（Gemma は利用許可が要るモデル。DP5・DP7 と同じトークン）。セル1で「シークレットへのアクセスを許可しますか？」が出たら、すぐに「許可」を押してね。
3. **「すべてのセルを実行」** でOK。初回は、本番前の確認とモデルのダウンロードに30〜50分、そのあと本番（セル7）が **8〜9時間くらい**、最後に監査（セル8）が1〜2時間です。終わると通知が来て、ランタイムは自動で切れます。
4. **途中で切れたとき**：もう一度 **「すべてのセルを実行」**。同じGPUの種類なら、セルRが前回の確認の記録を見つけて、確認をやり直さずに続きから再開します（できあがった仕事はDriveに残っていて、残りだけ計算します）。

- **通知**：スマホの ntfy アプリで `godpork-b06080cfb3cdfa8f` を購読しておくと、始まったとき・終わったとき・エラーで止まったときに届きます。ランタイムごと切れたときは届かないので、そのときは Colab を見てください。
- **終わったら自動でランタイムを切ります**（CU の節約）。Drive への保存を確定してから切るので、結果は残ります。エラーで止まったときは切らずに残します。切りたくないときは、最初に `AUTO_DISCONNECT = False` を実行してください。
- **エラーのとき**：通知の本文にエラーの文面（トレースバック）が入ります。全文は Drive の `DECISION_POINTS/_errors/` にも保存されるので、Claude が直接読めます。


**止まる場所（止まったら通知のエラー文をClaudeに見せてね）**
- セル2：インストール失敗／ゲート文書のハッシュ不一致
- セル2b：GPU版の迷いのサイン・素のサンプリング・自然な別の語の引き方・「確率が 10⁻⁶ に近い語」の選び方が、Pythonの実装と不一致（モデルなし）
- セル3：GSM-Symbolicの版がこれまでと違う／これまでの問題と重なる（トークンを作る前に止まります）
- セル4：E2　／　セル5b：G1・B1（まとめて計算する引き直しの確認）
- セル6：見込みが22時間超え
- セル7a／7：作業プロセスの失敗
- セル8：事後再生監査（土台の道と入れ替える語、引き直しの一部を作り直して照合）

問題の中身はどのセルも表示しません。

In [ ]:
# Cell 1 — mount Drive, check runtime, clone the exact frozen head.
from google.colab import drive, userdata
from pathlib import Path
import os, shutil, stat, subprocess, sys, json, time, hashlib, datetime as dt

# 通知（ntfy）：スマホの ntfy アプリでこの名前を購読しておくと、始まったとき・終わったとき・止まったときに届きます。エラーのときはエラーの文面も送ります（鍵・トークンらしい文字は伏せ字。全文は Drive の DECISION_POINTS/_errors）
import urllib.request
NTFY_TOPIC = 'godpork-b06080cfb3cdfa8f'
NB_LABEL = 'DP8 本番 A（Gemma・行0〜19、keep/swap/alt/swapt）'   # どのノートブックからの通知かの名札
def notify(msg, tags='pig'):
    try:
        req = urllib.request.Request('https://ntfy.sh/' + NTFY_TOPIC, data=('[' + NB_LABEL + '] ' + msg).encode('utf-8'),
                                     headers={'Title': 'GodPork - DP8 GPU A', 'Tags': tags})
        urllib.request.urlopen(req, timeout=10).read()
        print('通知を送りました：', msg)
    except Exception as e:
        print('通知は送れませんでした（実験には影響しません）：', type(e).__name__)
import os, re as _re, traceback as _tb, datetime as _edt
ERR_SLUG = 'DP8_GEMMA_A'   # Drive のエラーファイルの名前の頭
_SECRET_RE = _re.compile(r"sk-ant-[A-Za-z0-9_\-]+|github_pat_[A-Za-z0-9_]+|gh[pousr]_[A-Za-z0-9]{8,}|hf_[A-Za-z0-9]{8,}|x-access-token:[^@\s]+@")
def _scrub(s):
    s = _SECRET_RE.sub('[伏せ字]', s)
    return '\n'.join(l if len(l) <= 300 else l[:300] + ' …（長いので省略）' for l in s.splitlines())
def _error_text(result):
    e = result.error_in_exec if result.error_in_exec is not None else result.error_before_exec
    tb = ''.join(_tb.format_exception(type(e), e, e.__traceback__))
    first = (getattr(getattr(result, 'info', None), 'raw_cell', '') or '').strip().splitlines()[:1]
    return _scrub('セル: ' + (first[0][:100] if first else '?') + '\n' + tb)
def _save_error(text):
    base = '/content/drive/MyDrive'
    if not os.path.isdir(base):
        return None
    d = os.path.join(base, 'DECISION_POINTS', '_errors')
    os.makedirs(d, exist_ok=True)
    stem = ERR_SLUG + '_' + _edt.datetime.now(_edt.timezone(_edt.timedelta(hours=9))).strftime('%Y%m%d_%H%M%S')
    name, n = stem + '.txt', 1
    while os.path.exists(os.path.join(d, name)):
        n += 1
        name = stem + '_' + str(n) + '.txt'
    with open(os.path.join(d, name), 'w', encoding='utf-8') as fh:
        fh.write('[' + NB_LABEL + ']\n' + text)
    return 'DECISION_POINTS/_errors/' + name
def _short(text, limit=3300):
    b = text.encode('utf-8')
    if len(b) <= limit:
        return text
    return '…（前のほうは省略。全文は Drive）\n' + b[-limit:].decode('utf-8', 'ignore')
def _on_error(result):
    if result.error_in_exec is None and result.error_before_exec is None:
        return
    try:
        text = _error_text(result)
    except Exception as e2:
        text = 'エラーの文面を作れませんでした：' + type(e2).__name__
    try:
        where = _save_error(text)
    except Exception as e3:
        where = None
        text += '\n（Drive に保存できませんでした：' + type(e3).__name__ + '）'
    print('エラーの全文を通知に送ります' + ('（Drive：' + where + '）' if where else ''))
    notify('止まりました：エラーが出ています。' + ('全文は Drive の ' + where if where else 'Drive には保存できませんでした')
           + '\n\n' + _short(text), tags='warning')
if not globals().get('_HOOK_ON'):
    get_ipython().events.register('post_run_cell', _on_error); _HOOK_ON = True
notify('スタートしました（これが届けば通知の設定は OK です）', tags='rocket')

import tempfile

HALF = "A"     # "A" or "B"
drive.mount("/content/drive")
assert sys.version_info[:2] in ((3, 12), (3, 13)), f"CPython 3.12 or 3.13 required, got {sys.version}"
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True)
assert gpu.returncode == 0 and gpu.stdout.strip(), "GPU runtime required"
print("GPU:", gpu.stdout.strip())
import warnings
os.environ.pop("HF_HUB_DISABLE_IMPLICIT_TOKEN", None)   # gemma is gated: the token below must be sent
GPU_NAME = gpu.stdout.strip().split(",")[0]
if "L4" not in GPU_NAME:
    print("⚠️ このランタイムはL4ではありません（" + GPU_NAME + "）。動きますが、時間の見込みが変わります。")

EXPECTED_HEAD = "88b029da398556fc3b3c49903a318d7d5e81ba50"
BRANCH = "decision-points"
REPO_URL = "https://github.com/chichichi2001/Pork.git"
REPO = Path("/content/Pork")
THEME = Path("/content/drive/MyDrive/DECISION_POINTS")   # this theme's own folder (notebooks + outputs)
DP_ROOT = THEME / "dp8"
DP1_BASES = {h: THEME / "dp1" / ("DP1_1B_" + h + "_20261005") for h in ("A", "B")}   # READ-ONLY (overlap check)
DP2_RUN_BASES = {h: THEME / "dp2" / ("DP2_1B_" + h + "_20261006") for h in ("A", "B")}   # READ-ONLY (overlap check)
DP3_RUN_BASES = {h: THEME / "dp3" / ("DP3_1B_" + h + "_20261007") for h in ("A", "B")}   # READ-ONLY (overlap check)
DP4_RUN_BASES = {h: THEME / "dp4" / ("DP4_1B_" + h + "_20261008") for h in ("A", "B")}   # READ-ONLY (overlap check)
DP5_RUN_BASES = {h: THEME / "dp5" / ("DP5_GEMMA_" + h + "_20261008") for h in ("A", "B", "C")}   # READ-ONLY (overlap check)
DP7_RUN_BASES = {h: THEME / "dp7" / ("DP7_GEMMA_" + h + "_20261010") for h in ("A", "B")}   # READ-ONLY (overlap check)
PARTS = ("A", "B")
QP = Path("/content/drive/MyDrive/ULTIMAPORK/quality_pork")  # READ-ONLY: earlier studies' run identities (overlap check)
def run_base(half):
    return DP_ROOT / ("DP8_GEMMA_" + half + "_20261015")
BASE = run_base(HALF)
Q5_BASES = {h: QP / "q5" / ("QUALITY_PORK_Q5_" + h + "_20261004") for h in ("A", "B")}
Q3A_BASE = QP / "q3a_hightemp" / "QUALITY_PORK_Q3A_20261002"
Q3B_BASE = QP / "q3b_topk" / "QUALITY_PORK_Q3B_20261002"
Q4A_BASE = QP / "q4a_fence" / "QUALITY_PORK_Q4A_20261003"
Q4B_BASE = QP / "q4b_sc" / "QUALITY_PORK_Q4B_T1_20261003"
Q6_BASES = {h: QP / "q6s" / ("QUALITY_PORK_Q6S_" + h + "_20261004") for h in ("A", "B")}
Q8_BASES = {h: QP / "q8t" / ("QUALITY_PORK_Q8T_" + h + "_20261005") for h in ("A", "B")}
assert all((p / "RUN_IDENTITY.json").exists() for p in (Q3A_BASE, Q3B_BASE, Q4A_BASE, Q4B_BASE, *Q5_BASES.values(), *Q6_BASES.values(), DP1_BASES["A"], DP2_RUN_BASES["A"], DP3_RUN_BASES["A"], DP4_RUN_BASES["A"], *DP5_RUN_BASES.values(), *DP7_RUN_BASES.values())), "Q3-A・Q3-B・Q4-A・Q4-B・Q5・Q6S・DP1〜DP7の実行フォルダが見つかりません（データの版と、問題の重なりの確認に使います）"
UNITS = BASE / "units"
UNITS.mkdir(parents=True, exist_ok=True)
LOCKDIR = Path("/content/dp8_locks")   # LOCAL disk: all workers of this half run on this machine
MODEL_DIR = Path("/content/gemma-2-2b-it")   # LOCAL snapshot of the pinned revision (Cell 3)

def get_secret(*names):
    for name in names:
        try:
            value = userdata.get(name)
        except Exception:
            value = None
        if value and str(value).strip():
            return str(value).strip()
    return None

token = get_secret("GITHUB_TOKEN", "GH_TOKEN")
assert token, "Colab Secrets に GITHUB_TOKEN または GH_TOKEN を登録してください。"
askpass = Path("/tmp/pork_git_askpass.sh")
askpass.write_text('#!/bin/sh\ncase "$1" in\n  *Username*) echo "x-access-token" ;;\n  *Password*) echo "$GITHUB_TOKEN" ;;\n  *) echo "" ;;\nesac\n')
askpass.chmod(askpass.stat().st_mode | stat.S_IXUSR)
env = dict(os.environ, GITHUB_TOKEN=token, GIT_ASKPASS=str(askpass), GIT_TERMINAL_PROMPT="0")
if subprocess.run(["pgrep", "-f", "pork.dp8_runner"], capture_output=True).returncode == 0:
    print("🛑 このランタイムで DP8 の作業プロセスがまだ動いています。セル7の画面で終わるまで見守ってね（作り直さずに止めます）")
    raise SystemExit("workers are still running in this runtime — nothing was changed.")
os.chdir('/content')   # セル1 をやり直しても、消すフォルダの中にいないように（2026-10-14）
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(REPO)], check=True, env=env)
subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", EXPECTED_HEAD], check=True)
head = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
assert head == EXPECTED_HEAD, (head, EXPECTED_HEAD)
token = None
env.pop("GITHUB_TOKEN", None)
print("exact head:", head, "| part:", HALF)

# Hugging Face token (gemma is gated): secret HF_TOKEN; else the one already in this session; else a hidden input box.
# Kept only in this session's environment.
HF_STATUS = {"token_read": False, "source": None, "read_errors": []}
_tok = None
try:
    _v = userdata.get("HF_TOKEN")
    if _v and str(_v).strip():
        _tok, HF_STATUS["source"] = str(_v).strip(), "secret"
    else:
        HF_STATUS["read_errors"].append("HF_TOKEN: empty")
except Exception as e:
    HF_STATUS["read_errors"].append("HF_TOKEN: " + type(e).__name__)
if not _tok and os.environ.get("HF_TOKEN"):
    _tok, HF_STATUS["source"] = os.environ["HF_TOKEN"], "session"
if not _tok:
    from getpass import getpass
    print("👉 下の入力欄に、Hugging Face のトークン（hf_ で始まる文字列）を貼り付けて Enter を押してね（画面には表示されません）")
    _v = getpass("HF token: ")
    if _v and _v.strip():
        _tok, HF_STATUS["source"] = _v.strip(), "typed"
assert _tok, "Hugging Face のトークンが読めませんでした。左の鍵のマークで HF_TOKEN と「ノートブックからのアクセス」を確かめて、セル1をもう一度実行してね"
os.environ["HF_TOKEN"] = _tok
HF_STATUS["token_read"] = True
_tok = _v = None

# Self-check: this notebook is uploaded directly (not copied from GitHub by a loader), so verify that every other
# code cell is byte-identical to the version the builder produced.
NB_PATH = THEME / ("DP8_GEMMA_" + HALF + ".ipynb")
OTHER_CELLS_SHA = "f9b36304c31e1f981f9258e6563ede9c7d6d0b94fb832922743c8839da5f11f0"
if NB_PATH.exists():
    _codes = ["".join(c["source"]) for c in json.loads(NB_PATH.read_text())["cells"] if c["cell_type"] == "code"][1:]
    _got = hashlib.sha256("\n\x00\n".join(_codes).encode()).hexdigest()
    assert _got == OTHER_CELLS_SHA, ("このノートブックのセルがビルダーの版と違います — Claudeに見せてね", _got)
    print("notebook cells verified:", _got[:12])
else:
    print("⚠️ ノートブックが想定の場所（DECISION_POINTS フォルダ）にないので、セルの照合を飛ばしました。Claudeに伝えてね")


In [ ]:
# Cell 2 — install pinned packages and verify the frozen gate document.
EXPECTED_DOC_SHA = {
    "notes/dp8_rarity_bridge_gate.md": "881bb9bd0362e1e02ba3b6dff8134796569b9b1c3405f4cbdaf364a5c94560fc",
    "pork/pork_t_v1.py": "7e36a7a879490aa9918ec6db6c1193e747da3e8a51ee63fee7d6184206f47603"
}
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO),
                    "transformers==4.57.6", "huggingface_hub", "pyarrow", "scikit-learn", "scipy"],
                   capture_output=True, text=True)
(BASE / "install_log.txt").write_text(r.stdout + "\n--- STDERR ---\n" + r.stderr)
assert r.returncode == 0, r.stderr[-4000:]
for rel, want in EXPECTED_DOC_SHA.items():
    got = hashlib.sha256((REPO / rel).read_bytes()).hexdigest()
    assert got == want, (rel, got, want)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
from pork import dp2_real_forks as d2
from pork import dp8_bridge as d8
from pork import dp1_forks as dp
from pork import quality_pork_q8t as q8
assert d8.gate_doc_sha256(REPO) == d8.GATE_DOC_SHA256, "DP8 gate document differs from the frozen hash"
SPEC = d8.SPEC
# CPU unit tests (tests/test_dp8.py: items and exclusions incl. DP7 and CR-1's candidates, streams, the alt draw vs the
# plain-Python reference, work-pulling byte identity under shuffled claim orders and an interrupted-then-resumed run,
# alt first tokens, audit replay incl. a tampered w_tau and a tampered alt sample, analysis) were run before this
# notebook was built; the GPU-side parity check is Cell 2b.
import transformers, torch
assert transformers.__version__ == "4.57.6", transformers.__version__
print("install + gate hash OK; transformers", transformers.__version__, "torch", torch.__version__, "| half", HALF,
      "rows", d8.halves()[HALF][0], "...", d8.halves()[HALF][-1])

from pork.ultimapork_completion_c5a import PROMPTS as C5A_PROMPTS

def atomic_json(p, value):
    p.parent.mkdir(parents=True, exist_ok=True)
    data = json.dumps(value, ensure_ascii=False, indent=1, sort_keys=True) + "\n"
    fd, tmp = tempfile.mkstemp(prefix=p.name + ".", suffix=".tmp", dir=p.parent)
    with os.fdopen(fd, "w", encoding="utf-8") as f:
        f.write(data); f.flush(); os.fsync(f.fileno())
    os.replace(tmp, p)
    assert json.loads(p.read_text(encoding="utf-8")) == value

def need_cells(names, hint):
    missing = [n for n in names if n not in globals()]
    if missing:
        print("🛑 先に実行が必要なセルがあります：" + hint + "（足りないもの: " + ", ".join(missing) + "）")
        raise SystemExit("prerequisite cell not run — nothing was changed.")

def n_units():
    return len(list(UNITS.glob("base_*.json"))), len(list(UNITS.glob("point_*.json")))

def quarantine_dp8():
    """Move aside (rename, never delete) unit files of this half that do not validate, so they are recomputed."""
    from pork import dp8_runner as rr8
    moved = []
    qdir = BASE / "quarantine"
    for p in sorted(UNITS.glob("*.json")):
        parts = p.stem.split("_")
        kind, row = parts[0], int(parts[1])
        i = int(parts[2]) if kind == "point" else None
        if row not in d8.halves()[HALF]:
            continue
        try:
            rr8.read_envelope(p, kind, row, i, EXPECTED_HEAD, ITEMS_SHA, tries=2, wait=1.0)
        except (OSError, ValueError):
            qdir.mkdir(parents=True, exist_ok=True)
            p.rename(qdir / f"{p.name}.{int(time.time())}"); moved.append(p.name)
    return moved

RESUMED = False   # Cell R sets this when it finds a resumable run in this folder

In [ ]:
# Cell R — RESUME (automatic): if this run folder already holds a run identity and passing pre-evidence records from
# Cells 2b, 4, 5b and 6 for the same half, code commit and GPU type, Cells 2b-6 are skipped and the run continues in
# Cell 7 (finished units stay on Drive).  Otherwise (first run) this cell does nothing.
# It re-downloads the pinned model revision to local disk (the workers load it from there).
need_cells(["atomic_json", "C5A_PROMPTS", "GPU_NAME", "d2"], "セル1 → セル2")
RESUMED = False
# No model is loaded: it verifies that this run folder (same half, code commit and GPU type) already holds passing
# pre-evidence records from Cells 2b, 4, 5b and 6.  The worker count may differ (unit files do not depend on it).
import torch as torch_mod
from pork import kv_cached_backend as kv
from pork import dp8_runner as rr

id_path = BASE / "RUN_IDENTITY.json"
_pre = ("SAMPLER_PARITY.json", "E2.json", "G1.json")
_ident = json.loads(id_path.read_text()) if id_path.exists() else {}
_smokes = [json.loads(p.read_text()) for p in sorted(BASE.glob("TIMING_SMOKE_*.json"))]
_resumable = (bool(_ident) and _ident.get("gpu_name") == GPU_NAME and _ident.get("code_commit") == EXPECTED_HEAD
              and all((BASE / f).exists() for f in _pre) and any(s.get("gpu_name") == GPU_NAME for s in _smokes))
if not _resumable:
    print("初回（または別のGPU）なので、セル2bから順に確認します。")
else:
  identity = _ident
  rows_raw, DATA_ID = d2.load_rows_from_hub()
  ITEMS = d8.select_items(rows_raw)
  del rows_raw
  from huggingface_hub import snapshot_download   # a fresh VM has no local model copy: fetch the same pinned revision
  snapshot_download(SPEC["id"], revision=SPEC["revision"], local_dir=str(MODEL_DIR), token=os.environ["HF_TOKEN"],
                    allow_patterns=["*.json", "*.safetensors", "tokenizer*", "*.model"])
  assert identity.get("model_local_path") == str(MODEL_DIR) and identity.get("model_revision") == SPEC["revision"]
  ITEMS_SHA = d8.items_sha256(ITEMS)
  TEMPLATES = [it["template"] for it in ITEMS]
  gpu_gb = torch_mod.cuda.get_device_properties(0).total_memory / 1e9
  WORKERS = max(1, min(rr.MAX_WORKERS, kv.physical_cores(), int(gpu_gb // 10)))
  problems = [k for k, v in (("half", HALF), ("code_commit", EXPECTED_HEAD),
                             ("gate_doc_sha256", d8.GATE_DOC_SHA256), ("items_sha256", ITEMS_SHA),
                             ("model_revision", SPEC["revision"]), ("dataset_revision", DATA_ID["dataset_revision"]),
                             ("gpu_name", GPU_NAME))
              if identity.get(k) != v]
  if problems:
      print("🛑 セルRは使えません：前回の記録と今の状態が違います。ちがう項目:", problems,
            "（前回のGPU " + str(identity.get("gpu_name")) + "、今 " + GPU_NAME + "）")
      print("👉 前回と同じGPUの種類を選んで、セル1からやり直してね。")
      raise SystemExit("Cell R stopped (different machine) — nothing was changed.")

  def need(name):
      p = BASE / name
      if not p.exists():
          print(f"🛑 {name} がありません。セル2bから順に実行してください。")
          raise SystemExit("Cell R: pre-evidence record missing — nothing was changed.")
      return json.loads(p.read_text())
  parity, e2, g1 = need("SAMPLER_PARITY.json"), need("E2.json"), need("G1.json")
  assert parity.get("pass") is True and parity.get("gpu_name") == GPU_NAME, "サイン照合記録が不合格／別のGPU"
  assert len(e2) == 2 and all(r["argmax_agree"] for r in e2), "E2 記録が不合格"
  assert len(g1) == 3 and g1[2]["prompt"] == "B1_batched_vs_single" and all(r["steps"] > 0 and r["identical"] for r in g1), "G1/B1 記録が不合格"
  smokes = [json.loads(p.read_text()) for p in sorted(BASE.glob("TIMING_SMOKE_*.json"))]
  smokes = [s for s in smokes if s.get("gpu_name") == GPU_NAME]
  assert smokes and smokes[-1]["calibrated_hours"] <= d8.TIMING_STOP_HOURS, "見積もり記録がない／22時間超え"
  if identity.get("workers") != WORKERS:
      print("ℹ️ 並列数が前回（" + str(identity.get("workers")) + "）と違います（今回 " + str(WORKERS) + "）。結果のファイルは変わりません。")
  nb, npnt = n_units()
  atomic_json(BASE / f"RESUME_{int(time.time())}.json", {"half": HALF, "code_commit": EXPECTED_HEAD, "workers": WORKERS,
              "workers_at_identity": identity.get("workers"),
              "units_on_disk": {"base": nb, "point": npnt}, "prechecks_reused": ["SAMPLER_PARITY", "E2", "G1", "TIMING_SMOKE"]})
  print("RESUME OK — half", HALF, "| workers", WORKERS, "| units on disk: base", nb, "/", len(d8.halves()[HALF]),
        "points", npnt, "→ セル7a → セル7へ")
  RESUMED = True

In [ ]:
# Cell 2b — GPU vs plain-Python implementations on synthetic logits with ties (vocab 2,000 and 256,000): the per-step
# signals of the base path (top-1, top-2, p1, p2, raw entropy; DP1's step_signals), PURE sampling at T 1.0, the ALT
# first-token draw (as DP5: the base token removed and the rest renormalised; u near 1 included) and the w_tau choice
# (closest to tau = 1e-6 in log probability; top-1 and other ids excluded; ties to the smaller id; some steps hold
# several tokens exactly at tau).  No model.  Skipped when Cell R resumed this run.
if RESUMED:
    print('（再開：このセルの確認は前回の記録を使うので、飛ばします）')
else:
    need_cells(["atomic_json", "d2", "dp"], "セル1 → セル2")
    import math, random, struct
    import torch as torch_mod
    from pork import pork_t_v1 as pt
    from pork import q4b_samplers as qb

    Bg = pt.backend("torch", "cuda")

    def f32(x):
        return struct.unpack("f", struct.pack("f", x))[0]

    def synth(rng, vocab):
        xs = [f32(rng.gauss(0.0, rng.choice((1.0, 2.5, 4.0)))) for _ in range(vocab)]
        for i in range(0, vocab, 97):
            xs[i] = xs[0]
        if rng.random() < 0.3:
            j = rng.randrange(vocab); xs[j] = f32(xs[j] + 12.0)
        if rng.random() < 0.2:
            xs[1] = xs[0] = max(xs)        # exact tie for the top
        if rng.random() < 0.4:             # several tokens with the same logit near tau
            m = max(xs)
            v = f32(m + math.log(d8.TAU) + rng.choice((0.0, 0.3, -0.3)))
            for j in rng.sample(range(2, vocab), 3):
                xs[j] = v
        return xs

    rng = random.Random(d8.PARITY_SEED)
    names = ("SIGNALS", "PURE", "ALT", "TAU")
    total = {k: 0 for k in names}
    mism = {k: 0 for k in names}
    worst = {"p": 0.0, "entropy": 0.0, "tau_p_rel": 0.0}
    examples = []
    t0 = time.time()
    for vocab, steps in ((2000, 150), (256000, 6)):
        mg = mr = None
        for step in range(steps):
            xs = synth(rng, vocab)
            g = torch_mod.tensor(xs, dtype=torch_mod.float32, device="cuda")
            a = dp.step_signals(Bg, g, mg)
            b = dp.ref_step_signals(xs, mr)
            total["SIGNALS"] += 1
            worst["p"] = max(worst["p"], abs(a["p1"] - b["p1"]), abs(a["p2"] - b["p2"]))
            worst["entropy"] = max(worst["entropy"], abs(a["entropy"] - b["entropy"]))
            if ((a["top1"], a["top2"]) != (b["top1"], b["top2"])
                    or abs(a["p1"] - b["p1"]) > 1e-9 or abs(a["p2"] - b["p2"]) > 1e-9
                    or abs(a["entropy"] - b["entropy"]) > 1e-9):
                mism["SIGNALS"] += 1
                if len(examples) < 12:
                    examples.append({"vocab": vocab, "gpu": [a["top1"], a["top2"]], "ref": [b["top1"], b["top2"]]})
            mg, mr = a["mu_after"], b["mu_after"]
            u = rng.random()
            pa = tuple(int(v) for v in qb.pure_sample(Bg, g, u, d2.TEMPERATURE))
            pb = tuple(int(v) for v in qb.ref_pure(xs, u, d2.TEMPERATURE))
            total["PURE"] += 1
            if pa != pb:
                mism["PURE"] += 1
            exa = b["top1"] if step % 2 == 0 else rng.randrange(vocab)
            u2 = rng.random() if step % 7 else 0.9999999999
            aa, ab = d8.alt_sample(Bg, g, u2, d8.TEMPERATURE, exa), d8.ref_alt(xs, u2, d8.TEMPERATURE, exa)
            total["ALT"] += 1
            if aa != ab or aa == exa:
                mism["ALT"] += 1
            ex = rng.sample(range(vocab), 4)
            ta = d8.tau_choice(Bg, g, b["top1"], ex)
            tb = d8.ref_tau_choice(xs, b["top1"], ex)
            total["TAU"] += 1
            worst["tau_p_rel"] = max(worst["tau_p_rel"], abs(ta["p"] - tb["p"]) / tb["p"])
            if (ta["token"], ta["matched"]) != (tb["token"], tb["matched"]) or abs(ta["p"] - tb["p"]) > 1e-9 * tb["p"]:
                mism["TAU"] += 1
                if len(examples) < 12:
                    examples.append({"vocab": vocab, "tau_gpu": [ta["token"], ta["matched"]], "tau_ref": [tb["token"], tb["matched"]]})
    parity = {"total": total, "mismatches": mism, "worst_abs_diff": worst, "examples": examples,
              "seconds": time.time() - t0, "gpu_name": GPU_NAME, "code_commit": EXPECTED_HEAD,
              "pass": all(v == 0 for v in mism.values())}
    atomic_json(BASE / "SAMPLER_PARITY.json", parity)
    print(json.dumps(parity, indent=1))
    assert parity["pass"], "GPU signal parity FAILED — stop and show Claude."
    print("SIGNAL PARITY PASS")

In [ ]:
# Cell 3 — load the pinned GSM-Symbolic rows; select the 40 DP8 items (40 templates of 100; Q3-A ... Q8T, DP1, DP2/DP3
# (CT-1 = DP2's items), DP4, DP5 and DP7 items and all of CR-1's candidates (CR-1, CR-1a, PARA-1a) excluded; each
# earlier draw reconstructed and checked against its own run identity); download the pinned gemma-2-2b-it revision to local disk and load it (bfloat16, eager); record the ids
# excluded from w_tau (the run's EOS ids and the tokenizer's special ids); freeze this notebook's run identity (no
# generation).  Gate: on any data / overlap problem, stop here, before any benchmark token.
if RESUMED:
    print('（再開：このセルの確認は前回の記録を使うので、飛ばします）')
else:
    need_cells(["atomic_json", "C5A_PROMPTS", "GPU_NAME", "d2"], "セル1 → セル2")
    from pork import kv_cached_backend as kv
    from pork import dp8_runner as rr
    from pork.ct1_runner import load_gemma, torch_fns
    _par = BASE / "SAMPLER_PARITY.json"
    if not (_par.exists() and json.loads(_par.read_text()).get("pass") is True
            and json.loads(_par.read_text()).get("gpu_name") == GPU_NAME):
        print("🛑 先にこのGPUでセル2b（サインの照合）を実行してね。")
        raise SystemExit("Cell 3: no passing parity record for this GPU — nothing was changed.")
    try:
        rows_raw, DATA_ID = d2.load_rows_from_hub()
        ITEMS = d8.select_items(rows_raw)
        from pork import quality_pork_q3a_hightemp as q3a
        from pork import quality_pork_q3b_topk as q3b
        from pork import quality_pork_q4a_fence as q4a
        from pork import quality_pork_q4b_sc as q4b
        from pork import quality_pork_q5 as q5
        from pork import quality_pork_q6s as q6
        from pork import dp3_flip as d3
        from pork import dp4_weighting as d4w
        from pork import dp5_gemma as d5
        from pork import dp7_matched as d7
        runs = [("q3a", Q3A_BASE), ("q3b", Q3B_BASE), ("q4a", Q4A_BASE), ("q4b", Q4B_BASE), ("q5", Q5_BASES["A"]),
                ("q6s", Q6_BASES["A"]), ("dp1", DP1_BASES["A"])]
        if (Q8_BASES["A"] / "RUN_IDENTITY.json").exists():
            runs.append(("q8t", Q8_BASES["A"]))
        runs += [("dp2", DP2_RUN_BASES["A"]), ("dp3", DP3_RUN_BASES["A"]), ("dp4", DP4_RUN_BASES["A"]),
                 ("dp5", DP5_RUN_BASES["A"]), ("dp7", DP7_RUN_BASES["A"])]
        IDS = {name: json.loads((p / "RUN_IDENTITY.json").read_text()) for name, p in runs}
        assert json.loads((Q6_BASES["B"] / "RUN_IDENTITY.json").read_text())["items_sha256"] == IDS["q6s"]["items_sha256"]
        assert all(json.loads((p / "RUN_IDENTITY.json").read_text())["items_sha256"] == IDS["dp5"]["items_sha256"]
                   for p in DP5_RUN_BASES.values()), "DP5 parts disagree on items"
        assert all(json.loads((p / "RUN_IDENTITY.json").read_text())["items_sha256"] == IDS["dp7"]["items_sha256"]
                   for p in DP7_RUN_BASES.values()), "DP7 parts disagree on items"
        assert all(DATA_ID["dataset_revision"] == v["dataset_revision"] for v in IDS.values()), ("data revision differs", DATA_ID)
        mods = {"q3a": q3a, "q3b": q3b, "q4a": q4a, "q4b": q4b, "q5": q5, "q6s": q6, "q8t": q8, "dp1": dp, "dp2": d2, "dp3": d3,
                "dp4": d4w, "dp5": d5, "dp7": d7}
        _earlier = {name: mod.select_items(rows_raw) for name, mod in mods.items()}
        for name in IDS:
            assert mods[name].items_sha256(_earlier[name]) == IDS[name]["items_sha256"], f"{name} item reconstruction does not match its run"
        OVERLAP = {name: dp.overlap_with(ITEMS, its) for name, its in _earlier.items()}
        CR1_CAND_SHA = d8.check_cr1_candidates(rows_raw)          # anchored to CR-1's frozen candidate list
        OVERLAP["cr1_candidates"] = dp.overlap_with(ITEMS, d8.earlier_candidates(rows_raw))
        assert all(v == 0 for v in OVERLAP.values()), ("DP8 items overlap earlier items", OVERLAP)
        del rows_raw, _earlier
        ITEMS_SHA = d8.items_sha256(ITEMS)
    except Exception as exc:
        print("🛑 GSM-Symbolicの版・形式、または問題の重なりの確認で止まりました（ベンチマークのトークンはまだ作っていません）。")
        print("   画面をClaudeに見せてね。理由:", repr(exc)[:600])
        raise SystemExit("Cell 3: data check failed before any benchmark token.")
    TEMPLATES = [it["template"] for it in ITEMS]
    assert len(ITEMS) == d8.N_ITEMS and TEMPLATES == sorted(set(TEMPLATES))
    print("dataset:", DATA_ID, "| items:", len(ITEMS), "| overlap:", OVERLAP, "| Q8T identity checked:", "q8t" in IDS,
          "| items_sha256:", ITEMS_SHA)

    from huggingface_hub import snapshot_download
    snapshot_download(SPEC["id"], revision=SPEC["revision"], local_dir=str(MODEL_DIR), token=os.environ["HF_TOKEN"],
                      allow_patterns=["*.json", "*.safetensors", "tokenizer*", "*.model"])
    torch_mod, tokenizer, model = load_gemma(str(MODEL_DIR), None, d2.DEVICE)
    assert str(model.dtype) == "torch.bfloat16" and model.config._attn_implementation == "eager"
    EOS = kv.eos_ids(tokenizer, model)
    TAU_EXCLUDED = rr.excluded_ids(tokenizer, EOS)
    VOCAB = int(model.get_output_embeddings().weight.shape[0])
    assert all(0 <= x < VOCAB for x in TAU_EXCLUDED)
    _single, _batch = torch_fns(model, torch_mod, d2.DEVICE)
    gpu_gb = torch_mod.cuda.get_device_properties(0).total_memory / 1e9
    WORKERS = max(1, min(rr.MAX_WORKERS, kv.physical_cores(), int(gpu_gb // 10)))
    print("physical cores:", kv.physical_cores(), " GPU GB:", round(gpu_gb, 1), " workers:", WORKERS, " vocab:", VOCAB,
          " excluded from w_tau:", len(TAU_EXCLUDED))
    identity = {
        "schema": "dp8-run-identity-v1", "model": d8.MODEL, "half": HALF, "rows": list(d8.halves()[HALF]),
        "code_commit": EXPECTED_HEAD, "gate_doc_sha256": d8.GATE_DOC_SHA256,
        "overlap": OVERLAP, "q8t_identity_checked": "q8t" in IDS, "items_sha256": ITEMS_SHA,
        "model_id": SPEC["id"], "model_revision": SPEC["revision"], "dtype": SPEC["dtype"],
        "attn_implementation": SPEC["attn_implementation"], "model_local_path": str(MODEL_DIR),
        "dataset_id": dp.DATASET_ID, "dataset_revision": DATA_ID["dataset_revision"], "dataset_file": DATA_ID["dataset_file"],
        "eos_ids": list(EOS), "tau_excluded_ids": TAU_EXCLUDED, "tau": d8.TAU, "match_decades": d8.MATCH_DECADES,
        "vocab": VOCAB, "workers": WORKERS, "gpu_name": GPU_NAME,
        "python": sys.version.split()[0], "torch": torch_mod.__version__, "transformers": transformers.__version__,
        "numpy": __import__("numpy").__version__, "scipy": __import__("scipy").__version__,
        "scikit_learn": __import__("sklearn").__version__,
    }
    id_path = BASE / "RUN_IDENTITY.json"
    if id_path.exists():
        old = json.loads(id_path.read_text())
        drift = [k for k in ("half", "code_commit", "gate_doc_sha256", "items_sha256", "model_revision",
                             "dataset_revision", "eos_ids", "tau_excluded_ids", "vocab", "workers", "gpu_name")
                 if old.get(k) != identity[k]]
        no_units = not any(UNITS.glob("*.json"))
        if drift and set(drift) <= ({"workers", "gpu_name"} if no_units else {"workers"}):
            stamp = int(time.time())
            atomic_json(BASE / f"RUN_IDENTITY_superseded_{stamp}.json", old)
            for f in [*BASE.glob("E2.json"), *BASE.glob("G1.json"), *BASE.glob("TIMING_SMOKE_*.json")]:
                f.rename(BASE / f"superseded_{stamp}_{f.name}")
            atomic_json(id_path, identity)
            print("ℹ️ 前回とマシンが違うので（" + ", ".join(drift) + "）、このマシンで確認をやり直します。")
        elif drift:
            print("🛑 本番の途中で設定が変わっています:", drift, "— 前回と同じGPUの種類でセル1からやり直してね。")
            raise SystemExit("identity drift after benchmark units exist — nothing was changed.")
    else:
        atomic_json(id_path, identity)
    print(json.dumps({k: v for k, v in identity.items() if k != "tau_excluded_ids"}, indent=1))
    print("HF token source:", HF_STATUS["source"])

In [ ]:
# Cell 4 — E2: the DynamicCache step used for the base path (as CT-1) vs a full replay without cache, on the two consumed
# C5-A prompts (32 greedy steps).  The argmax must agree at every step (bfloat16: the logit difference is recorded only).
if RESUMED:
    print('（再開：このセルの確認は前回の記録を使うので、飛ばします）')
else:
    need_cells(["model", "tokenizer", "EOS", "VOCAB", "WORKERS"], "セル3")
    e2 = []
    for name, text in C5A_PROMPTS:
        ids = kv.chat_prompt_ids(tokenizer, text)
        step = _single()
        seq, past, new, worst, agree = list(ids), None, tuple(ids), 0.0, True
        for _ in range(dp.E2_STEPS):
            a, past = step(new, past)
            with torch_mod.no_grad():
                bfull = model(input_ids=torch_mod.tensor([seq], device=d2.DEVICE), use_cache=False).logits[0, -1].float()
            worst = max(worst, float((a - bfull).abs().max()))
            ta, tb = int(a.argmax()), int(bfull.argmax())
            agree = agree and ta == tb
            seq.append(tb); new = (tb,)
        e2.append({"prompt": name, "max_abs_logit_diff": worst, "argmax_agree": agree})
    atomic_json(BASE / "E2.json", e2)
    print(json.dumps(e2, indent=1))
    assert all(r["argmax_agree"] for r in e2), "E2 FAILED — stop."
    print("E2 PASS")

In [ ]:
# Cell 5b — G1: the DP8 base-path generator (DP7's) (step_signals' top-1 on the GPU, DynamicCache step, w_tau at every step)
# vs the reference greedy by full replay (64 tokens per consumed C5-A prompt).  B1: the batched step of 32 identical
# rows vs the single-row step (argmax agrees at >= 29 of 32 steps, as CT-1).  No benchmark tokens.
if RESUMED:
    print('（再開：このセルの確認は前回の記録を使うので、飛ばします）')
else:
    need_cells(["model", "tokenizer", "EOS", "VOCAB", "WORKERS", "TAU_EXCLUDED"], "セル3")
    from pork import pork_t_v1 as pt
    Bt = pt.backend("torch", d2.DEVICE)
    g1 = []
    for name, text in C5A_PROMPTS:
        ids = kv.chat_prompt_ids(tokenizer, text)
        ref_tokens, seq = [], list(ids)
        for _ in range(d8.G1_STEPS):
            with torch_mod.no_grad():
                tok = q8._greedy([float(v) for v in model(input_ids=torch_mod.tensor([seq], device=d2.DEVICE),
                                                           use_cache=False).logits[0, -1].float().tolist()])
            ref_tokens.append(tok); seq.append(tok)
            if tok in set(EOS): break
        gen = rr.generate_base_tau(Bt, _single(), ids, EOS, TAU_EXCLUDED, len(ref_tokens))
        base = gen["token_ids"]
        matched = sum(bool(s["tau_matched"]) for s in gen["steps"])
        g1.append({"prompt": name, "steps": len(ref_tokens), "identical": base == ref_tokens,
                   "tau_matched_steps": matched,
                   "tau_never_excluded_or_top1": all(s["tau_token"] not in set(TAU_EXCLUDED) and s["tau_token"] != s["top1"]
                                                     for s in gen["steps"])})
        assert g1[-1]["tau_never_excluded_or_top1"]
    ids = kv.chat_prompt_ids(tokenizer, C5A_PROMPTS[0][1])
    _bs, _ss = _batch(), _single()
    bp, sp, bn, sn = None, None, [tuple(ids)] * d8.K, tuple(ids)
    b1_agree = b1_rows = 0
    for _ in range(32):
        bl, bp = _bs(bn, bp); sl, sp = _ss(sn, sp)
        tok = int(sl.argmax()); b1_agree += int(int(bl[0].argmax()) == tok); b1_rows += int(bool((bl == bl[0:1]).all()))
        bn, sn = [(tok,)] * d8.K, (tok,)
    del _bs, _ss, bp, sp, bl, sl
    g1.append({"prompt": "B1_batched_vs_single", "steps": 32, "identical": b1_agree >= d8.B1_MIN_AGREE,
               "agree_steps": b1_agree, "rows_identical_steps": b1_rows})
    atomic_json(BASE / "G1.json", g1)
    print(json.dumps(g1, indent=1))
    assert all(r["identical"] for r in g1), "G1/B1 FAILED — stop."
    print("G1 PASS")

In [ ]:
# Cell 6 — timing smoke (not scored): per-step costs on the first C5-A prompt after a short warm-up.
# Gate stop line: the TYPICAL estimate (base path 230 tokens) x 2.7 per notebook over this machine's workers > 22 h.
# Typical per item: the base path (with w_tau at every step); at every position t = 0, 8, ... < 230: four batches of
# 32 (keep, swap, alt, swapt) of 230 - t steps, each after one prefill.
if RESUMED:
    print('（再開：このセルの確認は前回の記録を使うので、飛ばします）')
else:
    need_cells(["model", "tokenizer", "EOS", "VOCAB", "WORKERS", "TAU_EXCLUDED"], "セル3")
    from pork import pork_t_v1 as pt
    from pork.ct1_forks_replacement import generate_batch
    Bt = pt.backend("torch", d2.DEVICE)
    name, text = C5A_PROMPTS[0]
    ids = kv.chat_prompt_ids(tokenizer, text)
    no_eos = (-1,)
    rr.generate_base_tau(Bt, _single(), ids, no_eos, TAU_EXCLUDED, 8)  # warm-up
    n = q8.TIMING_SMOKE_STEPS
    per = {}
    t0 = time.time(); g = rr.generate_base_tau(Bt, _single(), ids, no_eos, TAU_EXCLUDED, n); per["BASE"] = (time.time() - t0) / n
    streams = [d8.stream("alt", 0, 0, j) for j in range(d8.K)]
    generate_batch(Bt, _batch(), ids, streams, no_eos, d8.TEMPERATURE, 4)
    t0 = time.time(); generate_batch(Bt, _batch(), ids, streams, no_eos, d8.TEMPERATURE, n); per["BATCH32"] = (time.time() - t0) / n
    long_ids = tuple(ids) + tuple(g["token_ids"]) * 12
    t0 = time.time(); generate_batch(Bt, _batch(), long_ids, streams, no_eos, d8.TEMPERATURE, 1); per["PREFILL"] = time.time() - t0
    KINDS_PER_POINT = len(d8.KINDS)
    # Memory probe (infrastructure, not scored): one batch of 32 at about prompt + 511 tokens, plus 4 decode steps.
    torch_mod.cuda.empty_cache(); torch_mod.cuda.reset_peak_memory_stats()
    _base_alloc = torch_mod.cuda.memory_allocated()
    probe_ids = (tuple(ids) + tuple(g["token_ids"]) * 64)[:len(ids) + d8.MAX_NEW_TOKENS - 1]
    generate_batch(Bt, _batch(), probe_ids, streams, no_eos, d8.TEMPERATURE, 4)
    peak_gb = (torch_mod.cuda.max_memory_allocated() - _base_alloc) / 1e9 + _base_alloc / 1e9
    torch_mod.cuda.empty_cache()
    print(f"memory probe: peak {peak_gb:.1f} GB for one worker's model + a 32-row batch at {len(probe_ids)} tokens")

    def per_item(L):
        rest = [L - t for t in range(0, L, d8.STRIDE)]
        return L * per["BASE"] + sum(rest) * KINDS_PER_POINT * per["BATCH32"] + len(rest) * KINDS_PER_POINT * per["PREFILL"]

    rows = d8.halves()[HALF]
    typical_hours = len(rows) * per_item(d8.TYPICAL_BASE_TOKENS) / WORKERS / 3600
    calibrated_hours = typical_hours * d8.TIMING_FACTOR
    smoke = {"half": HALF, "per_step_s": per, "workers": WORKERS, "typical_total_hours": typical_hours,
             "calibrated_hours": calibrated_hours, "factor": d8.TIMING_FACTOR, "stop_hours": d8.TIMING_STOP_HOURS,
             "typical_base_tokens": d8.TYPICAL_BASE_TOKENS, "units_done": list(n_units()), "rows": len(rows),
             "gpu_name": GPU_NAME, "peak_gb_32rows": peak_gb}
    atomic_json(BASE / f"TIMING_SMOKE_{int(time.time())}.json", smoke)
    print(json.dumps(smoke, indent=1))
    assert calibrated_hours <= d8.TIMING_STOP_HOURS, "Projected runtime exceeds 22 h — stop and show Claude."
    print(f"TIMING OK（見込みは約{calibrated_hours:.1f}時間。これまでの実際の時間で直した値です）")

In [ ]:
# Cell 7a — pre-token infrastructure blocks (Q2-A v1 rule, as in Q2-B..DP5: "a caught failure before the first
# benchmark token of an invocation is an infrastructure block and may be retried unchanged").  A worker marker
# qualifies ONLY if its traceback is a cuBLAS handle-creation failure (cublasCreate runs at the process's first GPU
# matmul, i.e. before any token of that invocation).  Qualifying markers are renamed (kept, not deleted) and logged;
# any other marker still stops the run.
need_cells(["atomic_json", "BASE"], "セル1 → セル2")
import re
qualifying = "when calling `cublasCreate(handle)`"
log_path = BASE / "INFRA_BLOCKS.json"
log_entries = json.loads(log_path.read_text()) if log_path.exists() else []
for marker in sorted(BASE.glob("RUN_FAILED_worker*.txt")):
    text = marker.read_text(encoding="utf-8", errors="replace")
    worker = int(re.search(r"worker(\d+)", marker.name).group(1))
    if qualifying in text and "Traceback" in text:
        new_name = f"INFRA_BLOCK_worker{worker}_{int(time.time())}.txt"
        marker.rename(BASE / new_name)
        log_entries.append({"worker": worker, "renamed_to": new_name, "reason": "cublasCreate failure = first GPU call",
                            "sha256": hashlib.sha256(text.encode()).hexdigest(), "unix": time.time()})
        print(f"worker {worker}: pre-token infrastructure block -> {new_name}")
    else:
        print(f"worker {worker}: NOT a provable pre-token failure — stop and show Claude.")
atomic_json(log_path, log_entries)
left = sorted(p.name for p in BASE.glob("RUN_FAILED*.txt"))
assert not left, ("PROTOCOL_FAIL markers remain — do not continue; show Claude", left)
print("no failure markers remain")

In [ ]:
# Cell 7 — BENCHMARK (evidence): WORKERS processes of pork.dp8_runner pull unfinished units of this half (local lock
# files; every unit's file depends only on its row, point and sample index, so the claim order cannot change any file).
# Workers are launched one at a time only while the GPU has room for another model copy.
# Resumable: after any disconnect, rerun Cells 1, 2, R, 7a, 7 — finished units stay on Drive and are not recomputed.
need_cells(["WORKERS", "ITEMS_SHA", "torch_mod"], "セルR（またはセル3）")
import gc, signal
for marker in [BASE / "RUN_FAILED.txt", *BASE.glob("RUN_FAILED_worker*.txt")]:
    if marker.exists():
        raise SystemExit(f"{marker.name} exists (PROTOCOL_FAIL). Do not continue; show Claude.")
for _name in ("model", "fast", "full", "runner", "backend", "b", "g", "gen", "Bt", "lf", "_single", "_batch", "_bs", "_ss", "bl", "sl", "bp", "sp", "a", "bfull", "step", "past"):
    globals().pop(_name, None)  # this process keeps no model; each worker loads its own
gc.collect(); torch_mod.cuda.empty_cache()
HALF_ROWS = len(d8.halves()[HALF])

def free_gb():
    out = subprocess.run(["nvidia-smi", "--query-gpu=memory.free", "--format=csv,noheader,nounits"],
                         capture_output=True, text=True, check=True).stdout
    return int(out.strip().splitlines()[0]) / 1024

def live_worker_pids():
    r = subprocess.run(["pgrep", "-f", "pork.dp8_runner --half " + HALF], capture_output=True, text=True)
    return [int(x) for x in r.stdout.split()]

def pid_alive(pid):
    try:
        os.kill(pid, 0)
        return True
    except OSError:
        return False

old = live_worker_pids()
procs = {}
if old:
    print("ℹ️ 前に起動した作業プロセスがまだ動いています（pid " + str(old) + "）。二重起動せず、終わるまで見守ります。")
else:
    _free, _total = (x / 1e9 for x in torch_mod.cuda.mem_get_info())
    assert _free > _total - 1.5, f"GPU memory still held by this notebook ({_total - _free:.1f} GB) — show Claude"
    shutil.rmtree(LOCKDIR, ignore_errors=True)   # no live worker: every lock is stale (its unit has no file yet)
    from pork import dp8_runner as rr
    moved = quarantine_dp8()
    if moved:   # e.g. a file cut off when an earlier VM died: moved aside (kept) and recomputed
        qlog = BASE / "QUARANTINE.json"
        entries = json.loads(qlog.read_text()) if qlog.exists() else []
        entries.append({"unix": time.time(), "moved": moved})
        atomic_json(qlog, entries)
    MARGIN_GB = 1.5
    per_worker_gb = 10.0  # conservative (gemma-2-2b bf16 ~5.2 GB + KV cache of 32 samples up to ~750 tokens)
    _sm = [json.loads(p.read_text()) for p in sorted(BASE.glob("TIMING_SMOKE_*.json"))]
    _sm = [x for x in _sm if x.get("gpu_name") == GPU_NAME and x.get("peak_gb_32rows")]
    if _sm:   # Cell 6's measured peak (+15 %, + ~1 GB of CUDA context) if it is larger than the floor
        per_worker_gb = max(per_worker_gb, _sm[-1]["peak_gb_32rows"] * 1.15 + 1.0)
    print(f"planning {per_worker_gb:.1f} GB per worker")
    for w in range(WORKERS):
        if any(BASE.glob("RUN_FAILED*.txt")):
            print("a failure marker appeared — not launching more workers"); break
        if free_gb() < per_worker_gb + MARGIN_GB:
            print(f"GPU has only {free_gb():.1f} GB free; running with {len(procs)} workers"); break
        before = free_gb()
        logf = BASE / f"worker_{w}.log"
        start_size = len(logf.read_text(errors="replace")) if logf.exists() else 0
        procs[w] = subprocess.Popen(
            [sys.executable, "-m", "pork.dp8_runner", "--half", HALF, "--worker", str(w), "--base", str(BASE),
             "--lockdir", str(LOCKDIR), "--head", EXPECTED_HEAD],
            cwd=REPO, stdout=open(logf, "a"), stderr=subprocess.STDOUT, env=dict(os.environ, PYTHONUNBUFFERED="1"))
        print(f"worker {w}: pid {procs[w].pid} (free before {before:.1f} GB)", flush=True)
        t = time.time()
        while procs[w].poll() is None and "READY" not in logf.read_text(errors="replace")[start_size:] and time.time() - t < 900:
            time.sleep(5)
        time.sleep(45)  # let the first forward passes allocate their buffers
        if procs[w].poll() is None:
            used = before - free_gb()
            per_worker_gb = max(used * 1.15, per_worker_gb) if used > 0 else per_worker_gb
            print(f"worker {w}: running; ~{used:.1f} GB used, planning {per_worker_gb:.1f} GB per worker", flush=True)
    assert procs, "no worker could be started — show Claude"

def stop_all(why):
    print("🛑", why, "— 作業プロセスを止めます。Claudeに見せてね", flush=True)
    for p in procs.values():
        if p.poll() is None:
            p.send_signal(signal.SIGTERM)
    for pid in old:
        if pid_alive(pid):
            os.kill(pid, signal.SIGTERM)

t0 = time.time()
failed = None
STALL_MIN = 60
last_change, last_counts = time.time(), n_units()
while True:
    running = [w for w, p in procs.items() if p.poll() is None] + [pid for pid in old if pid_alive(pid)]
    bad = [w for w, p in procs.items() if p.poll() not in (None, 0)]
    if any(BASE.glob("RUN_FAILED*.txt")) or bad:
        failed = "worker failure (" + str(bad or sorted(p.name for p in BASE.glob("RUN_FAILED*.txt"))) + ")"
        stop_all(failed); break
    if not running:
        break
    time.sleep(60)
    nb, npnt = n_units()
    if (nb, npnt) != last_counts:
        last_change, last_counts = time.time(), (nb, npnt)
    elif time.time() - last_change > STALL_MIN * 60:
        failed = f"no unit finished for {STALL_MIN} min (stalled)"
        stop_all(failed); break
    print(f"{(time.time()-t0)/60:6.1f} min  base {nb}/{HALF_ROWS}  points {npnt}  running {len(running)}", flush=True)
for w, p in procs.items():
    p.wait()
    print(f"worker {w}: exit code {p.returncode}")
    print((BASE / f"worker_{w}.log").read_text()[-1200:])
assert failed is None, failed
from pork import dp8_runner as rr
CASES_HALF = rr.load_cases(d8.halves()[HALF], units_dir=UNITS, code_commit=EXPECTED_HEAD, items_hash=ITEMS_SHA)
nb, npnt = n_units()
print("complete: base", nb, "/", HALF_ROWS, "| points", npnt, "👉 次はセル8（監査）")

In [ ]:
# Cell 8 — post-hoc replay audit for THIS notebook's rows (gate notes/dp8_rarity_bridge_gate.md): the pre-declared items
# of this part (5 in all; seed 202610151607): the base path with the CPU argmax over the logits (top-1 and top-2) and
# w_tau re-derived by the plain-Python reference at every swapped position, and at 3 points the keep/swap/alt/swapt
# batches recomputed as in the run with samples j = 0..3 drawn by the reference (the alt first token too; NumPy
# stable-sort reference, as the gate names).  The pass rule (<= 1 diverged item) is applied in DP8_ANALYZE.
need_cells(["ITEMS_SHA", "d8"], "セルR（またはセル3）")
assert not any(BASE.glob("RUN_FAILED*.txt")), "failure markers exist — show Claude"
if not (BASE / f"AUDIT_{HALF}.json").exists():
    logf = BASE / "audit.log"
    p = subprocess.run([sys.executable, "-m", "pork.dp8_runner", "--half", HALF, "--worker", "99", "--base", str(BASE),
                        "--lockdir", str(LOCKDIR), "--head", EXPECTED_HEAD, "--audit", "--fast-audit-reference"],
                       cwd=REPO, stdout=open(logf, "a"), stderr=subprocess.STDOUT, env=dict(os.environ, PYTHONUNBUFFERED="1"))
    print(logf.read_text()[-600:])
    assert p.returncode == 0, "監査の実行に失敗しました — Claudeに見せてね\n" + logf.read_text()[-3000:]
aud = json.loads((BASE / f"AUDIT_{HALF}.json").read_text())
assert aud["audit_reference"] == "numpy_stable_argsort", aud["audit_reference"]
print(json.dumps({k: aud[k] for k in ("half", "rows", "diverged_rows", "audit_reference")}))
print("✅ この本は終わり。もう1本も終わったら、DP8_ANALYZE（CPU）で集計します。Claudeに知らせてね")
notify('終わりました：本番と監査（ずれた問題 ' + str(aud['diverged_rows']) + '）。もう1本も終わったら集計へ', tags='tada')
if globals().get('AUTO_DISCONNECT', True):
    print('Drive への書き込みを確定してから、ランタイムを切ります（CU の節約。止めたいときは AUTO_DISCONNECT = False）')
    from google.colab import drive as _drive, runtime as _runtime
    _drive.flush_and_unmount()
    time.sleep(3)
    _runtime.unassign()